In [23]:
import nbimporter
from test_copy import load_and_display_heads, reduce_tracking_data_direct
game_data, play_data, player_play_data, player_data, tracking_data = load_and_display_heads()
reduced_tracking_data = reduce_tracking_data_direct(tracking_data, 30)

Game Data Head:
       gameId  season  week   gameDate gameTimeEastern homeTeamAbbr  \
0  2022090800    2022     1   9/8/2022        20:20:00           LA   
1  2022091100    2022     1  9/11/2022        13:00:00          ATL   
2  2022091101    2022     1  9/11/2022        13:00:00          CAR   
3  2022091102    2022     1  9/11/2022        13:00:00          CHI   
4  2022091103    2022     1  9/11/2022        13:00:00          CIN   

  visitorTeamAbbr  homeFinalScore  visitorFinalScore  
0             BUF              10                 31  
1              NO              26                 27  
2             CLE              24                 26  
3              SF              19                 10  
4             PIT              20                 23  

Play Data Head:
       gameId  playId                                    playDescription  \
0  2022102302    2655  (1:54) (Shotgun) J.Burrow pass short middle to...   
1  2022091809    3698  (2:13) (Shotgun) J.Burrow pass shor

In [ ]:
play_data["gameId"]

KeyError: 'GameId_x'

In [4]:
play_data = play_data.merge(reduced_tracking_data, on='playId', how='left')
# print(play_data)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

def generate_x_data(data):
    
    x_data = []
    # Step 2: Normalize numerical features (e.g., x, y, speed, acceleration)
    numerical_columns = ['x', 'y', 's', 'a', 'yardsToGo', 'down', 'preSnapHomeScore', 'preSnapVisitorScore', 'quarter']
    scaler = StandardScaler()
    data[numerical_columns] = scaler.fit_transform(data[numerical_columns])

    # Step 3: One-hot encode categorical features (playDirection, event, position)
    # data = pd.get_dummies(data, columns=['playDirection', 'event', 'position'], drop_first=False)
    
    # Step 4: Handle missing values (e.g., filling with zeros)
    data = data.fillna(0)
    
    # Iterate through plays
    for play_id, play_df in data.groupby("playId"):
        for _, play_info in play_df.iterrows():
            
            # Extract situation features
            situation_features = {
                "gameId": play_info["gameId"],
                "down": play_info["down"],
                "distance": play_info["yardsToGo"],
                "score_differential": play_info["preSnapHomeScore"] - play_info["preSnapVisitorScore"],
                "quarter": play_info["quarter"],
                # "game_clock": play_info["gameClock"]
            }

            # Get player tracking data at the snap
            # snap_frame = play_df[play_df["frameType"] == "SNAP"]
            # if snap_frame.empty:
            #     continue
            # for _, row in snap_frame.iterrows():
            # player_features = {
            #     "x": play_info["x"],
            #     "y": play_info["y"],
            #     "s": play_info["s"],
            #     "dir": play_info["dir"],
            #     "o": play_info["o"],
            #     "team": play_info["club"],
            #     # "player_role": play_info["position"]
            # }
            # Create X (features) for the play
            x_data.append({
                "situation": situation_features,
                # "players": player_features
            })
    return x_data

In [6]:
data = play_data
x_data = generate_x_data(data)
# print(x_data)

In [8]:
x_data[0]

{'situation': {'down': -0.9601565808511344,
  'distance': 0.4062463714471245,
  'score_differential': -0.0984403821407569,
  'quarter': -1.3094143544723622},
 'players': {'x': 0.0, 'y': 0.0, 's': 0.0, 'dir': 0.0, 'o': 0.0, 'team': 0}}

In [24]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

def convert_game_clock_to_seconds(game_clock):
    """Convert gameClock (MM:SS) to seconds left in the quarter."""
    minutes, seconds = map(int, game_clock.split(':'))
    return minutes * 60 + seconds

def generate_x_data(play_data, game_data):
    """Generate features for plays by merging play_data and game_data and extracting relevant features."""
    # Merge play_data with game_data to include homeTeamAbbr, visitorTeamAbbr, and win probability
    data = play_data.merge(
        game_data[['gameId', 'homeTeamAbbr', 'visitorTeamAbbr']],
        on='gameId',
        how='left'
    )

    # Calculate score differential for the possession team
    data['score_differential'] = np.where(
        data['possessionTeam'] == data['homeTeamAbbr'],
        data['preSnapHomeScore'] - data['preSnapVisitorScore'],
        data['preSnapVisitorScore'] - data['preSnapHomeScore']
    )

    data['win_probability'] = np.where(
        data['possessionTeam'] == data['homeTeamAbbr'],
        data['preSnapHomeTeamWinProbability'],
        1 - data['preSnapHomeTeamWinProbability']
    )

    # Convert gameClock to seconds left in the quarter
    data['gameClockSeconds'] = data['gameClock'].apply(convert_game_clock_to_seconds)

    # Normalize numerical features
    numerical_columns = ['yardsToGo', 'down', 'preSnapHomeScore', 'preSnapVisitorScore', 'quarter', 'gameClockSeconds']
    scaler = StandardScaler()
    data[numerical_columns] = scaler.fit_transform(data[numerical_columns])

    # Handle missing values
    data = data.fillna(0)

    # Extract relevant features for each play
    features = []
    for play_id, play_df in data.groupby("playId"):
        for _, play_info in play_df.iterrows():
            # Extract situation features
            situation_features = {
                "gameId": play_info["gameId"],
                "playId": play_info["playId"],
                "quarter": play_info["quarter"],
                "down": play_info["down"],
                "yardsToGo": play_info["yardsToGo"],
                "possessionTeam": play_info["possessionTeam"],
                "defensiveTeam": play_info["defensiveTeam"],
                "absoluteYardlineNumber": play_info["absoluteYardlineNumber"],
                "gameClockSeconds": play_info["gameClockSeconds"],
                "offenseFormation": play_info["offenseFormation"],
                "receiverAlignment": play_info["receiverAlignment"],
                "playClockAtSnap": play_info.get("playClockAtSnap", 0),
                "score_differential": play_info["score_differential"],
                "win_probability": play_info["win_probability"]
            }

            # Append situation features to the list
            features.append(situation_features)

    return pd.DataFrame(features)

# Example usage
# play_data = pd.read_csv('play_data.csv')
# game_data = pd.read_csv('game_data.csv')
# features = generate_x_data(play_data, game_data)
# print(features.head())


In [29]:
x_data = generate_x_data(play_data, game_data)

In [31]:
x_data.to_csv("../data/x_data.csv", encoding='utf-8', index=False)

,gameId,playId,quarter,down,yardsToGo,possessionTeam,defensiveTeam,absoluteYardlineNumber,gameClockSeconds,offenseFormation,receiverAlignment,playClockAtSnap,score_differential,win_probability
0,2022100200,54,-1.384795,-0.968115,0.393677,MIN,NO,35,1.727920,SINGLEBACK,2x2,5.0,0,0.602240
1,2022092512,54,-1.384795,-0.968115,0.393677,TB,GB,35,1.727920,SINGLEBACK,2x2,5.0,0,0.586653
2,2022101300,54,-1.384795,-0.968115,0.393677,CHI,WAS,85,1.727920,SHOTGUN,3x1,11.0,0,0.446686
3,2022091110,55,-1.384795,-0.968115,0.393677,KC,ARI,85,1.727920,SHOTGUN,2x2,13.0,0,0.683450
4,2022091500,55,-1.384795,-0.968115,0.393677,KC,LAC,85,1.727920,SHOTGUN,3x1,8.0,0,0.704539
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16119,2022091103,5015,2.138291,0.240829,0.393677,PIT,CIN,30,-1.393637,SHOTGUN,3x1,12.0,0,0.404982
16120,2022091103,5039,2.138291,1.449774,-1.908975,PIT,CIN,39,-1.456142,SHOTGUN,3x1,32.0,0,0.417455
16121,2022091103,5074,2.138291,-0.968115,0.393677,PIT,CIN,65,-1.489233,SHOTGUN,3x1,13.0,0,0.605415
16122,2022091103,5096,2.138291,0.240829,0.393677,PIT,CIN,65,-1.503940,SHOTGUN,2x2,6.0,0,0.593763
